# corrector_04 - The corrector on real OCR errors (role B)

Needs: `corrector_02` and `corrector_03` already run (`ngram.pkl.gz`, `viterbi.json`, `tagger.pt` in
`checkpoints/corrector/`) and role A's `outputs/ocr_out.jsonl`.

**Only the val split is used.** The test split is scored once, by role C, at the very end.

1. Measure the real OCR diacritic errors on val (what kind, how often) and check the label style.
2. Turn them into noise probabilities and a Viterbi emission probability.
3. Fine-tune the transformer on that noise (GPU) and calibrate it again.
4. Run every corrector on the real OCR val text: CER / WER before and after, fixed / broken.
5. Error analysis: what gets fixed, what gets broken, what is missed.

Runtime: **T4 GPU** for step 3 (the other steps also run on CPU). Needs the `GITHUB_TOKEN` secret.

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

token = userdata.get("GITHUB_TOKEN")
BRANCH = "b/corrector"  # change to "main" after the code is merged
!rm -rf /content/repo
!git clone -q -b {BRANCH} https://{token}@github.com/wickyhien18/Vietnamese-Handwriting-OCR.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

DRIVE_ROOT = "/content/drive/MyDrive/Vietnamese-Handwriting-OCR"
DATA_DIR = f"{DRIVE_ROOT}/data"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
OUT_DIR = f"{DRIVE_ROOT}/outputs"

In [ ]:
!python -m unittest discover -s tests -t . 2>&1 | tail -4

In [ ]:
import itertools, json, os, time
import torch

OCR_OUT_PATH = f"{OUT_DIR}/ocr_out.jsonl"
MODEL_DIR = f"{CKPT_DIR}/corrector"
CORPUS_DIR = f"{DATA_DIR}/corpus"
TAUS = [0.0, 0.5, 0.7, 0.9]       # confidence thresholds to look at (role C chooses the final one)
FT_SENTENCES = 400_000            # fine-tuning sentences with the OCR-like noise
FT_EPOCHS = 1
FT_LR = 1e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

from src.corrector.ocr_adapt import ocr_lines
with open(OCR_OUT_PATH, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
val_lines = ocr_lines(rows, split="val")
print(len(rows), "OCR rows,", len(val_lines), "val lines used (test is not touched)")

## 1. What do the real OCR errors look like?

In [ ]:
from src.corrector.ngram import NgramModel
from src.corrector.ocr_adapt import DIACRITIC_KINDS, BASE, SAME, ocr_error_stats

stats = ocr_error_stats(val_lines)
pairs = stats["pairs"]
print(f"aligned syllable pairs: {pairs}  (label syllables: {stats['label_syllables']})")
for kind in (SAME,) + DIACRITIC_KINDS + (BASE,):
    print(f"  {kind:<16}{stats[kind]:>7}  {stats[kind] / pairs:7.2%}")
wrong = pairs - stats[SAME]
fixable = sum(stats[k] for k in DIACRITIC_KINDS)
print(f"wrong syllables: {wrong}, of which a diacritic corrector could fix {fixable} ({fixable / wrong:.1%})")
print(f"label syllables with the tone in the other position (hoà instead of hòa): "
      f"{stats['label_other_tone_style']} ({stats['label_other_tone_style'] / pairs:.2%})")

ngram = NgramModel.load(f"{MODEL_DIR}/ngram.pkl.gz")
from src.corrector.ngram import sentence_words
from src.common.text_utils import strip_diacritics
label_words = [w for gt, _ in val_lines for w in sentence_words(gt)]
known = sum(w in ngram.unigram for w in label_words) / len(label_words)
known_key = sum(strip_diacritics(w) in ngram.forms for w in label_words) / len(label_words)
print(f"coverage of the label syllables by the corpus: {known:.2%} (plain form known: {known_key:.2%})")

## 2. Noise probabilities and Viterbi emission from the real errors

In [ ]:
from dataclasses import asdict
from src.corrector.noise import NoiseConfig
from src.corrector.ocr_adapt import noise_config_from_stats, p_same_from_stats

ocr_noise = noise_config_from_stats(stats)
p_same_ocr = p_same_from_stats(stats)
with open(f"{MODEL_DIR}/viterbi.json", encoding="utf-8") as f:
    viterbi_settings = json.load(f)
print("synthetic noise (before):", asdict(NoiseConfig()))
print("OCR-like noise (after):  ", {k: round(v, 4) for k, v in asdict(ocr_noise).items()})
print(f"Viterbi p_same: synthetic {viterbi_settings['p_same']:.4f} -> real OCR {p_same_ocr:.4f}")

with open(f"{MODEL_DIR}/ocr_noise.json", "w", encoding="utf-8") as f:
    json.dump({"noise": asdict(ocr_noise), "p_same": p_same_ocr, "k": viterbi_settings["k"],
               "stats": dict(stats)}, f, ensure_ascii=False, indent=2)
print("saved", f"{MODEL_DIR}/ocr_noise.json")

## 3. Fine-tune the transformer on the OCR-like noise (GPU)

In [ ]:
import numpy as np
from torch.utils.data import DataLoader
from src.corrector.calibration import calibration_report, fit_temperature
from src.corrector.tagger import (DiacriticTagger, NoisySentences, collate, collect_logits,
                                  make_scheduler, save_checkpoint, train_epoch)
from src.corrector.tagger_data import TaggerVocab

ckpt = torch.load(f"{MODEL_DIR}/tagger.pt", map_location=device, weights_only=False)
CONFIG = ckpt["config"]
vocab = TaggerVocab.from_dict(ckpt["vocab"])
model = DiacriticTagger(len(vocab.forms), len(vocab.keys), **CONFIG).to(device)
model.load_state_dict(ckpt["state_dict"])

def read_lines(path, start, n):
    with open(path, encoding="utf-8") as f:
        return [line.strip() for line in itertools.islice(f, start, start + n)]

# Sentences after the first 600k, so the fine-tuning text is new to the model where possible.
ft_sentences = read_lines(f"{CORPUS_DIR}/train.txt", 600_000, FT_SENTENCES)
val_sentences = read_lines(f"{CORPUS_DIR}/val.txt", 0, 5_000)
ft_data = NoisySentences(ft_sentences, vocab, ocr_noise, seed=7, max_len=CONFIG["max_len"])
val_data = NoisySentences(val_sentences, vocab, ocr_noise, seed=123, max_len=CONFIG["max_len"], fixed_noise=True)
ft_loader = DataLoader(ft_data, batch_size=128, shuffle=True, collate_fn=collate, num_workers=2)
val_loader = DataLoader(val_data, batch_size=256, collate_fn=collate, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=0.01)
scheduler = make_scheduler(optimizer, FT_EPOCHS * len(ft_loader), warmup_steps=200)
K_MAX = vocab.k + 1
for epoch in range(FT_EPOCHS):
    ft_data.epoch = epoch
    start = time.time()
    loss = train_epoch(model, ft_loader, optimizer, scheduler, device, log_every=1000)
    print(f"fine-tune epoch {epoch + 1}: loss {loss:.4f} ({time.time() - start:.0f} s)")

logits, targets = collect_logits(model, val_loader, device, K_MAX)
multi = np.isfinite(logits).sum(axis=1) >= 2
temperature = fit_temperature(logits[multi], targets[multi])
calib_ocr = calibration_report(logits[multi], targets[multi], temperature)
print(json.dumps(calib_ocr, indent=2))
save_checkpoint(f"{MODEL_DIR}/tagger_ocr.pt", model, vocab, CONFIG, temperature=temperature)
print("saved", f"{MODEL_DIR}/tagger_ocr.pt")

## 4. Every corrector on the real OCR val text

CER and WER: before (OCR) -> after (corrected). fixed / broken are counted on aligned syllables
(option 2 of the evaluation notebook). broken rate = broken / syllables the OCR had right.
fixed/fixable = share of the diacritic-only OCR errors that got fixed.

In [ ]:
from src.corrector import api
from src.corrector.correctors import ViterbiCorrector
from src.corrector.ocr_adapt import evaluate_on_ocr, print_ocr_table
from src.corrector.viterbi import Emission

correctors = {
    "identity": api.load(method="identity"),
    "frequency": api.load(MODEL_DIR, method="frequency"),
    "viterbi": api.load(MODEL_DIR, method="viterbi"),
    "viterbi (OCR p_same)": ViterbiCorrector(ngram, Emission(p_same_ocr), viterbi_settings["k"]),
    "transformer": api.load(MODEL_DIR, method="transformer", device=device),
    "transformer (OCR ft)": api.load(MODEL_DIR, method="transformer", device=device,
                                     tagger_file="tagger_ocr.pt"),
}
real_results = {}
for name, corrector in correctors.items():
    start = time.time()
    real_results[name] = evaluate_on_ocr(corrector, val_lines, tau=0.0)
    print(f"{name}: {time.time() - start:.0f} s")
print()
print_ocr_table(real_results)

### Confidence threshold: accept a suggestion only when conf >= tau (role C picks the final tau)

In [ ]:
best_two = sorted((n for n in real_results if n != "identity"),
                  key=lambda n: real_results[n]["cer_after"])[:2]
tau_results = {}
for name in best_two:
    for tau in TAUS:
        tau_results[f"{name} tau={tau}"] = evaluate_on_ocr(correctors[name], val_lines, tau=tau)
print_ocr_table(tau_results)

with open(f"{OUT_DIR}/corrector_real_ocr_eval.json", "w", encoding="utf-8") as f:
    strip = lambda r: {k: v for k, v in r.items() if k != "examples"}
    json.dump({"split": "val", "lines": len(val_lines), "stats": dict(stats),
               "ocr_noise": asdict(ocr_noise), "p_same_ocr": p_same_ocr, "calibration_ocr_ft": globals().get("calib_ocr"),
               "results": {k: strip(v) for k, v in real_results.items()},
               "tau_sweep": {k: strip(v) for k, v in tau_results.items()}}, f, ensure_ascii=False, indent=2)
print("\nsaved", f"{OUT_DIR}/corrector_real_ocr_eval.json")

## 5. Error analysis (for the report)

In [ ]:
BEST = best_two[0]
r = real_results[BEST]
print("best method on val by CER:", BEST)
print("\nmost frequent FIXED   (OCR -> label):")
for (ocr_s, gt_s), n in r["examples"]["fixed"]:
    print(f"  {n:>4}  {ocr_s} -> {gt_s}")
print("\nmost frequent BROKEN  (label -> output):")
for (gt_s, out_s), n in r["examples"]["broken"]:
    print(f"  {n:>4}  {gt_s} -> {out_s}")
print("\nmost frequent MISSED  (OCR -> output, label):  diacritic-only errors left wrong")
for (ocr_s, out_s, gt_s), n in r["examples"]["missed"]:
    print(f"  {n:>4}  {ocr_s} -> {out_s}   (label {gt_s})")

In [ ]:
import random
corrector = correctors[BEST]
for gt, ocr in random.Random(0).sample(val_lines, 8):
    out = corrector.correct(ocr)
    print("label :", gt)
    print("OCR   :", ocr)
    print("output:", out["text"])
    print("conf  :", [round(c, 2) for c in out["conf"]])
    print()